# pix2pix

---
## 目的
pix2pix [1]を構築し，ペア画像（対になる2枚の画像，例えば「セグメンテーションラベル画像」と「対応する実写真」）を用いた画像変換（image-to-image translation）の仕組みを理解する．画像同士の画素単位の対応関係を直接利用することで，比較的シンプルな仕組みで変換を学習できる点に注目する．

[1] Phillip Isola, Jun-Yan Zhu, Tinghui Zhou and Alexei A. Efros, "Image-to-Image Translation with Conditional Adversarial Nets," CVPR, 2017.


## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import os
import random
import zipfile
import tarfile
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセット
pix2pixの学習には，変換前後で画素単位に対応が取れた「ペア画像」が必要です．
本ノートブックでは，建物のラベル画像（セグメンテーションラベル）から，実際の建物の写真を生成するfacadesデータセットを使用します．
他のデータセットを使いたい場合は，[配布URL一覧](https://efrosgans.eecs.berkeley.edu/pix2pix/datasets/)から選択し，`wget`で取得してください．

In [ ]:
data_root = './facades'
if not os.path.isdir(data_root):
    gdown.download(id='1vLdgS3AveIvj1IljGg7eArvqGMq0sQOv', output='facades.tar.gz', quiet=False)
    with tarfile.open('facades.tar.gz') as f:
        f.extractall()

### DataLoaderの定義
facadesデータセットは，1枚の画像ファイルの中に，写真（左半分）とラベル画像（右半分）が横に連結された状態で格納されています．`Pix2PixDataset`では，これを読み込んだのち中央で分割し，`(ラベル画像, 写真)`のペアとして返します．

ペア画像を用いる都合上，データ拡張（左右反転）は**写真とラベル画像の両方に全く同じ変換を適用する**必要があります．2枚の画像にそれぞれ独立にランダムな変換を適用してしまうと，写真とラベル画像とで反転の有無が食い違い，画素の対応関係が崩れてしまいます．そのため，`transforms.Compose`は使わず，1つの乱数の結果を両方の画像に適用します．


In [ ]:
class Pix2PixDataset(torch.utils.data.Dataset):
    def __init__(self, root, augment=True):
        self.root = root
        self.files = sorted(os.listdir(root))
        self.augment = augment
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.root, self.files[idx])).convert('RGB')
        w, h = img.size
        half = w // 2
        photo = img.crop((0, 0, half, h))
        label = img.crop((half, 0, w, h))

        if self.augment and random.random() > 0.5:  # 写真・ラベル画像に同じ左右反転を適用する
            photo = photo.transpose(Image.FLIP_LEFT_RIGHT)
            label = label.transpose(Image.FLIP_LEFT_RIGHT)

        photo = self.normalize(self.to_tensor(photo))
        label = self.normalize(self.to_tensor(label))
        return label, photo  # (入力: ラベル画像, 目標: 写真)


train_data = Pix2PixDataset('facades/train', augment=True)
val_data = Pix2PixDataset('facades/val', augment=False)
print(len(train_data), len(val_data))

label, photo = train_data[0]
print('label:', label.shape, 'photo:', photo.shape)

## ネットワークの構築
### Generator（U-Net）
画像変換を行うGeneratorの最も単純な構成は，入力画像をEncoderで一度低解像度の特徴に圧縮してからDecoderで復元する，Encoder-Decoder構造です．しかし，この構造では，Encoderで一度失われた低レベルな情報（物体の輪郭やエッジの位置など）を，Decoderだけで完全に復元するのは困難です．

pix2pixのGeneratorは，**U-Net**と呼ばれる構造を採用しています．U-Netは，Encoder-Decoder構造に加えて，Encoderの各層の出力を，対応する解像度のDecoderの層へチャネル方向に連結する**skip connection**を持ちます．これにより，Decoderは低解像度の特徴だけでなく，Encoderが保持していた高解像度の情報も利用でき，入力画像の構造（線や輪郭など）を保ったまま変換を行うことができます．

U-Netは，同じ構造（Downsampling 1層 + Upsampling 1層 + skip connection）を入れ子状に繰り返す構造をしているため，`UNetBlock`を再帰的に組み合わせることでネットワーク全体を構築します．画像サイズ$256\times256$に対して，8回のダウンサンプリング（$256\to128\to\cdots\to1$）を行います．


In [ ]:
class UNetBlock(nn.Module):
    """U-Netの1階層分（Downsampling 1層 + Upsampling 1層）を表すモジュール．
    submoduleにさらに内側のUNetBlockを再帰的に渡すことで，全体としてU字型のネットワークを構築する．"""
    def __init__(self, outer_ch, inner_ch, in_ch=None, submodule=None, outermost=False, innermost=False, use_dropout=False):
        super().__init__()
        self.outermost = outermost
        in_ch = outer_ch if in_ch is None else in_ch
        downconv = nn.Conv2d(in_ch, inner_ch, kernel_size=4, stride=2, padding=1, bias=False)

        if outermost:  # 最も外側：入力画像を受け取り，最終的な出力画像を返す
            upconv = nn.ConvTranspose2d(inner_ch * 2, outer_ch, kernel_size=4, stride=2, padding=1)
            model = [downconv, submodule, nn.ReLU(inplace=True), upconv, nn.Tanh()]
        elif innermost:  # 最も内側（ボトルネック）：skip connectionの相手がいない
            upconv = nn.ConvTranspose2d(inner_ch, outer_ch, kernel_size=4, stride=2, padding=1, bias=False)
            model = [nn.LeakyReLU(0.2, inplace=True), downconv,
                     nn.ReLU(inplace=True), upconv, nn.BatchNorm2d(outer_ch)]
        else:  # 中間層
            upconv = nn.ConvTranspose2d(inner_ch * 2, outer_ch, kernel_size=4, stride=2, padding=1, bias=False)
            model = [nn.LeakyReLU(0.2, inplace=True), downconv, nn.BatchNorm2d(inner_ch),
                     submodule,
                     nn.ReLU(inplace=True), upconv, nn.BatchNorm2d(outer_ch)]
            if use_dropout:
                model.append(nn.Dropout(0.5))

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        return torch.cat([x, self.model(x)], dim=1)  # skip connection（チャネル方向に連結）


class UNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64):
        super().__init__()
        block = UNetBlock(ngf * 8, ngf * 8, innermost=True)             # 1x1（ボトルネック）
        block = UNetBlock(ngf * 8, ngf * 8, submodule=block, use_dropout=True)  # 2x2
        block = UNetBlock(ngf * 8, ngf * 8, submodule=block, use_dropout=True)  # 4x4
        block = UNetBlock(ngf * 4, ngf * 8, submodule=block)             # 8x8
        block = UNetBlock(ngf * 2, ngf * 4, submodule=block)             # 16x16
        block = UNetBlock(ngf, ngf * 2, submodule=block)                 # 32x32
        block = UNetBlock(out_ch, ngf, in_ch=in_ch, submodule=block, outermost=True)  # 256x256（入出力）
        self.model = block

    def forward(self, x):
        return self.model(x)

### Discriminator（Conditional PatchGAN）
Discriminatorには，**PatchGAN**と呼ばれる構造を使用します．`dcgan.ipynb`などの通常のDiscriminatorは，1枚の画像全体に対して「本物らしさ」を表す1つのスカラー値を出力しますが，PatchGANは，画像を$N\times N$個の小さなパッチ（局所領域）に分割し，パッチごとに「本物らしさ」を判定します．実装上は，全結合層を使わずに畳み込み層のみを重ね，最終的な出力を1つの値ではなく$N\times N$の特徴マップとして出力させます．出力マップの各画素が，入力画像上のある局所領域（パッチ）に対応する「本物らしさ」の判定値を表します．

画像全体を1つの値で判定するのではなく，局所的なパッチ単位で判定することで，Discriminatorは画像の細部の質感やエッジといった高周波成分（局所的なテクスチャ）に注目しやすくなり，Generatorはより鮮明で細部の整った画像を生成するように学習が進みます．また，Discriminatorが軽量になる（全結合層を持たないため，入力解像度によらずパラメータ数が変わらない）という利点もあります．

本ノートブックのpix2pixでは，これに加えて，**入力画像と出力（または正解）画像をチャネル方向に連結してから入力する**条件付き（Conditional）構造とします．これにより，Discriminatorは「その画像が本物らしいか」だけでなく，「入力画像の内容と対応しているか」も判定するようになります．

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_ch=6, n_layers=3):  # in_ch: 入力画像+出力画像のチャネル数（3+3=6）
        super().__init__()
        out_features = 64
        modules = [
            nn.Conv2d(in_ch, out_features, kernel_size=4, stride=2, padding=1, bias=True),
            nn.LeakyReLU(negative_slope=0.2, inplace=True)]

        for i in range(n_layers):
            in_features = out_features
            out_features = in_features * 2
            stride = 1 if i == n_layers - 1 else 2
            modules += [
                nn.Conv2d(in_features, out_features, kernel_size=4, stride=stride, padding=1, bias=True),
                nn.BatchNorm2d(out_features),
                nn.LeakyReLU(negative_slope=0.2, inplace=True)]

        modules += [nn.Conv2d(out_features, 1, kernel_size=4, stride=1, padding=1, bias=True)]
        self.layers = nn.Sequential(*modules)

    def forward(self, input_img, target_img):
        x = torch.cat([input_img, target_img], dim=1)  # 条件付け：入力画像と出力（または正解）画像を連結する
        return self.layers(x)

## ネットワークの作成，学習に必要なパラメータの定義
pix2pixは，入力画像（ラベル画像）から出力画像（写真）への一方向の変換のみを学習するため，GeneratorとDiscriminatorはそれぞれ1つずつで済みます．


In [ ]:
lr = 0.0002
betas = (0.5, 0.999)
batch_size = 1
n_epochs = 100
lambda_l1 = 100  # L1 lossの重み

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

G = UNetGenerator(in_ch=3, out_ch=3).to(device)
D = Discriminator(in_ch=6, n_layers=3).to(device)

g_opt = optim.Adam(G.parameters(), lr=lr, betas=betas)
d_opt = optim.Adam(D.parameters(), lr=lr, betas=betas)

adv_loss = nn.BCEWithLogitsLoss().to(device)
l1_loss = nn.L1Loss().to(device)

## pix2pixの学習
pix2pixの誤差関数は，以下の2種類から構成されます．

* **GAN loss**：通常のConditional GANと同じ誤差関数です．Discriminatorは，`(入力画像, 正解の写真)`のペアを本物，`(入力画像, Generatorが生成した写真)`のペアを偽物と判定するように学習し，Generatorはその逆を狙います．
$$\mathcal{L}_{GAN}(G, D) = \mathbb{E}\left[\log D(x, y)\right] + \mathbb{E}\left[\log(1 - D(x, G(x)))\right]$$
* **L1 loss**：ペア画像が使えることを活かし，生成画像と正解画像の画素値の差（L1距離）を直接小さくする誤差です．
$$\mathcal{L}_{L1}(G) = \mathbb{E}\left[\|y - G(x)\|_{1}\right]$$

Generator全体の誤差は，$\mathcal{L}_{GAN}(G, D) + \lambda_{L1}\mathcal{L}_{L1}(G)$です．L1 lossは，画像全体をぼやけさせてでも正解に近づけようとする性質があり，GAN lossは，ぼやけた画像よりも鮮明な画像を好む性質があります．`lambda_l1`を大きくすることで，pix2pixはこの両方をバランスさせて学習します．


In [ ]:
for epoch in range(1, n_epochs + 1):
    G.train()
    D.train()
    for label_img, photo_img in train_loader:
        label_img, photo_img = label_img.to(device), photo_img.to(device)

        fake_photo = G(label_img)

        # Discriminatorの更新
        d_opt.zero_grad()
        out_real = D(label_img, photo_img)
        out_fake = D(label_img, fake_photo.detach())
        d_loss = (adv_loss(out_real, torch.ones_like(out_real)) + adv_loss(out_fake, torch.zeros_like(out_fake))) * 0.5
        d_loss.backward()
        d_opt.step()

        # Generatorの更新
        g_opt.zero_grad()
        out_fake = D(label_img, fake_photo)
        g_adv_loss = adv_loss(out_fake, torch.ones_like(out_fake))
        g_l1_loss = l1_loss(fake_photo, photo_img)
        g_loss = g_adv_loss + lambda_l1 * g_l1_loss
        g_loss.backward()
        g_opt.step()

    print(f'epoch: {epoch} | D loss: {d_loss.item():.4f} | '
        f'G loss: {g_loss.item():.4f} (adv: {g_adv_loss.item():.4f}, L1: {g_l1_loss.item():.4f}) |')

    # 50 epochsに１回モデルを保存
    if epoch % 50 == 0:
        torch.save(G.state_dict(), "pix2pix_gen_%03d.pt" % epoch)
        torch.save(D.state_dict(), "pix2pix_dis_%03d.pt" % epoch)

## 学習済みモデルのロード
pix2pixはネットワークが深いため，綺麗な変換をするためには長時間の学習が必要です．演習時間内に学習を終えることが難しいため，学習を割愛し，学習済みモデルを用いて変換結果を確認します（ご興味のある方は，講義終了後にご自身で動かしてみてください）．約10~15分ほどかかります．

以下でダウンロードする学習済みモデルには，実際の建物の写真を生成するfacadesデータセットに対する学習モデルが含まれています．

In [ ]:
data_root = './pix2pix_model'
if not os.path.isdir(data_root):
    gdown.download(id='1VP04D0x1OSA3u7ot14cFSxiuUdMUyvld', output='pix2pix_model.zip', quiet=False)
    with zipfile.ZipFile('pix2pix_model.zip') as f:
        f.extractall('./')

G.load_state_dict(torch.load('./pix2pix_model/pix2pix_gen_100.pt', map_location=device))
D.load_state_dict(torch.load('./pix2pix_model/pix2pix_dis_100.pt', map_location=device))

## 変換結果の確認
評価用データ（`facades/val`）から数枚選び，「入力（ラベル画像）」「生成された写真」「正解の写真」を並べて可視化します．

In [ ]:
def denormalize(tensor_img):
    return (tensor_img * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).cpu().numpy()


G.eval()
n_show = 4
fig, axes = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
with torch.no_grad():
    for i in range(n_show):
        label_img, photo_img = val_data[i]
        fake_photo = G(label_img.unsqueeze(0).to(device)).squeeze(0)

        axes[i, 0].imshow(denormalize(label_img)); axes[i, 0].axis('off')
        axes[i, 1].imshow(denormalize(fake_photo)); axes[i, 1].axis('off')
        axes[i, 2].imshow(denormalize(photo_img)); axes[i, 2].axis('off')

axes[0, 0].set_title('input (label)')
axes[0, 1].set_title('generated (fake photo)')
axes[0, 2].set_title('ground truth (real photo)')
plt.tight_layout()
plt.show()

## 課題

1. `lambda_l1`の値を大きく・小さく変更して学習し，生成される画像の鮮明さやラベル画像との対応関係がどのように変化するか確認してください．
2. [facades以外のpix2pix用データセット](https://efrosgans.eecs.berkeley.edu/pix2pix/datasets/)（例：`maps`, `edges2shoes`）に変更して学習してみましょう．
3. 本ノートブックのpix2pix（ペア画像が必要）と`cycle_gan.ipynb`のCycleGAN（ペア画像が不要）を比較し，それぞれの利点・欠点を整理してください．